In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, mean_absolute_error


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:

# Task 1: Write your code here:
file1_path= os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(file1_path) # read the data

In [ ]:
# Task 2: Write your code here:
print(f"Shape: {df.shape}") # shows number of rows and cols
df.head() # print first five rows

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe() # get statistical of the numerical data

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_clean = df.drop( 'Order_ID', axis=1)
print(df_clean.shape) # to show that the col drop

In [ ]:
 # Task 2: Write your code here:
print("Missing values:")
print(df_clean.isnull().sum()) # to know which col has missing values

In [ ]:
df_clean.dropna(inplace=True) # change the df_clean
print(df_clean.isnull().sum()) # to check if the data is clean or not

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")
check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
categ_col= df_clean.select_dtypes(include=["object"]).columns
print("Categorical Columns:", list(categ_col)) # get categorical col
label_encoders = {}
for col in categ_col:
  le = LabelEncoder()
  df_clean[col] = le.fit_transform(df_clean[col])
  label_encoders[col] = le
df_clean

In [ ]:
# Task 5: Write your code here:
all_cols= df_clean.columns
all_cols= all_cols[:-1]
scaler = StandardScaler()
df_clean[all_cols] = scaler.fit_transform(df_clean[all_cols])
df_clean.head()

In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop("Delivery_Time", axis=1).astype(float)
y = df_clean['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import  RandomForestRegressor
kf = KFold(n_splits=5, shuffle=True, random_state=42)
model=RandomForestRegressor(n_estimators=320, max_depth=4)
mae_scores=[]
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
# indexing for each fold
  X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
  y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  mae = mean_absolute_error(y_test, y_pred)
  mae_scores.append(mae)
print(f"the averaged MAE: {sum(mae_scores)/len(mae_scores):,.2f}")

In [ ]:
# Task 1: Write your code here:

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(y_pred, bins=30, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here:
from catboost import CatBoostRegressor
models = {'RandomForestRegressor':RandomForestRegressor(n_estimators=320, max_depth=4),
          'CatBoostRegressor': CatBoostRegressor(verbose=0)}
all_results = {}
for name in models:
  all_results[name] = {'mae': []}
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]
  for model_name, model in models.items():
    print(f"Training {model_name}...")
    model.fit(X_train, y_train)
# Predict
    y_pred = model.predict(X_test)
# Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)
    all_results[model_name]["mae"].append(mae)
for model_name in all_results:
  print(f" MAE: {np.mean(all_results[model_name]['mae']):.4f}")